# 01 Data Check

Maal:
- Laste inn data fra Snowflake + vaer
- Kjoere preprocessing
- Sjekke datakvalitet foer modellering
- Verifisere nyhetsbrev_b7-feature

In [1]:
from pathlib import Path
import sys

import pandas as pd

project_root = Path.cwd().resolve().parent.parent
code_dir = project_root / "code"
if str(code_dir) not in sys.path:
    sys.path.append(str(code_dir))

from utils.data_load import load_raw_data
from utils.data_prep import data_prep

In [2]:
# Konfig
NEWSLETTER_CSV = project_root / "data" / "newsletter_sendouts.csv"

# Last data
session = None  # Sett session = get_or_create_session() hvis du vil gjenbruke samme session

df_inngang, df_info, df_weather = load_raw_data(
    session=session,
    weather_path=project_root / "data" / "oslo_weather.csv",
)

print("Raw shapes")
print("df_inngang:", df_inngang.shape)
print("df_info:", df_info.shape)
print("df_weather:", df_weather.shape)
print("newsletter csv finnes:", NEWSLETTER_CSV.exists())

Raw shapes
df_inngang: (157333, 14)
df_info: (911, 108)
df_weather: (731, 5)
newsletter csv finnes: True


In [3]:
# Bygg analyseklar dataframe
cols_to_remove = []

df = data_prep(
    df_inngang=df_inngang,
    df_info=df_info,
    df_weather=df_weather,
    cols_to_remove=cols_to_remove,
    newsletter_dates_csv=str(NEWSLETTER_CSV),
)

print("Prepared shape:", df.shape)
print("Antall kolonner:", len(df.columns))
print("Dato fra/til:", df["ankomst_dato"].min(), "->", df["ankomst_dato"].max())

Prepared shape: (500, 132)
Antall kolonner: 132
Dato fra/til: 2024-06-03 -> 2026-06-02


In [4]:
# Manglende verdier (topp 25)
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
missing.head(25)

helligdag                         480
antall_hf_b30_epf01_ny             44
antall_nye_kunder_f7_epf01_ny      44
antall_hf_b7_epf01_ny              44
antall_hf_b30_tot                  44
antall_nye_kunder_b30_tot          44
antall_nye_kunder_b30_ny           44
antall_hf_b30_ny                   44
antall_hf_f7_epf01_ny              44
antall_nye_kunder_b30_epf01_ny     44
antall_nye_kunder_f30_epf01_ny     44
antall_hf_f30_epf01_ny             44
antall_nye_kunder_b7_epf01_ny      44
dtype: int64

In [5]:
# Kontroll av sentrale maalkolonner
check_cols = [
    "antall_samtaler",
    "behandlingstid_snitt",
    "total_behandlingstid",
    "nyhetsbrev_b7",
]

stats = df[check_cols].describe(include="all").T
stats

,count,mean,std,min,25%,50%,75%,max
antall_samtaler,500.0,314.666000,101.619208,3.000000,240.750000,330.000000,380.000000,591.000000
behandlingstid_snitt,500.0,299.404732,37.655162,15.666667,275.197512,294.596277,319.207895,440.672043
total_behandlingstid,500.0,93411.588000,29079.229718,47.000000,75109.500000,98307.500000,112923.250000,189454.000000
nyhetsbrev_b7,500.0,0.232000,0.422532,0.000000,0.000000,0.000000,0.000000,1.000000


In [6]:
# Validering av nyhetsbrev_b7
print("Unike verdier i nyhetsbrev_b7:", sorted(df["nyhetsbrev_b7"].dropna().unique().tolist()))
print("Andel dager med nyhetsbrev_b7=1:", round((df["nyhetsbrev_b7"] == 1).mean(), 4))

df[["ankomst_dato", "nyhetsbrev_b7"]].head(15)

Unike verdier i nyhetsbrev_b7: [0, 1]
Andel dager med nyhetsbrev_b7=1: 0.232


,ankomst_dato,nyhetsbrev_b7
0,2024-06-03,1
1,2024-06-04,1
2,2024-06-05,1
3,2024-06-06,1
4,2024-06-07,1
5,2024-06-10,1
6,2024-06-11,1
7,2024-06-12,1
8,2024-06-13,1
9,2024-06-14,1
